In [2]:
from flask import Flask, request, jsonify
import pandas as pd
import joblib
import os

app = Flask(__name__)

# Load model safely
model_path = "model.pkl"
model = joblib.load(model_path)

@app.route('/')
def home():
    return "AI Market Intelligence API Running 🚀"

@app.route('/predict', methods=['POST'])
def predict():

    data = request.json   # ✅ THIS LINE IS MUST

    input_df = pd.DataFrame([data])
    
    # Feature engineering
    input_df['price_diff'] = input_df['price'] - input_df['competitor_price']
    input_df['prev_sales'] = 50
    input_df['rolling_avg'] = 50
    input_df['demand_trend'] = 0
    
    input_df = pd.get_dummies(input_df)
    
    for col in model.feature_names_in_:
        if col not in input_df.columns:
            input_df[col] = 0
    
    input_df = input_df[model.feature_names_in_]
    
    prediction = model.predict(input_df)[0]

    # 🧠 RECOMMENDATION ENGINE
    recommendations = []

    price = data['price']
    comp_price = data['competitor_price']
    discount = data['discount']
    stock = data['stock_estimate']

    if price < comp_price:
        recommendations.append("You can increase price slightly")
    elif price > comp_price:
        recommendations.append("Reduce price (Competitor cheaper)")

    if discount > 0.3:
        recommendations.append("High discount may reduce profit")

    if prediction > 3:
        recommendations.append("High demand expected → Increase stock")
    elif prediction < 1:
        recommendations.append("Low demand → Avoid overstock")

    if not recommendations:
        recommendations.append("Pricing and strategy look optimal")

    return jsonify({
        "predicted_demand": float(prediction),
        "recommendations": recommendations
    })
    return jsonify({
        "predicted_demand": float(prediction)
    })

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\chand\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
pip install flask-cors

Note: you may need to restart the kernel to use updated packages.


In [4]:
from flask_cors import CORS

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import joblib

# Initialize app
app = Flask(__name__)
CORS(app)

# ✅ Load trained model
model = joblib.load("../Backend/model.pkl")


# Home route
@app.route('/')
def home():
    return "AI Market Intelligence API Running 🚀"


# Prediction route
@app.route('/predict', methods=['POST'])
def predict():

    # =========================
    # STEP 1: Get data
    # =========================
    data = request.json

    # =========================
    # STEP 2: Create DataFrame
    # =========================
    input_df = pd.DataFrame([data])

    # =========================
    # STEP 3: Feature Engineering
    # =========================
    input_df['price_diff'] = input_df['price'] - input_df['competitor_price']
    input_df['prev_sales'] = 50
    input_df['rolling_avg'] = 50
    input_df['demand_trend'] = 0

    # =========================
    # STEP 4: Encoding
    # =========================
    input_df = pd.get_dummies(input_df)

    # =========================
    # STEP 5: Match model columns
    # =========================
    for col in model.feature_names_in_:
        if col not in input_df.columns:
            input_df[col] = 0

    input_df = input_df[model.feature_names_in_]

    # =========================
    # STEP 6: Prediction (IMPORTANT)
    # =========================
    prediction = model.predict(input_df)[0]

    # =========================
    # STEP 7: Extract variables
    # =========================
    price = data['price']
    comp_price = data['competitor_price']
    discount = data['discount']

    # =========================
    # STEP 8: Demand Level
    # =========================
    if prediction > 3:
        demand_level = "High 🔥"
    elif prediction > 1:
        demand_level = "Medium 📊"
    else:
        demand_level = "Low ⚠️"

    # =========================
    # STEP 9: Recommendation Engine
    # =========================
    recommendations = []

    # Basic rules
    if price < comp_price:
        recommendations.append("Increase price slightly")
    elif price > comp_price:
        recommendations.append("Reduce price (Competitor cheaper)")

    if discount > 0.3:
        recommendations.append("High discount may reduce profit")

    # Advanced scoring system
    score = 0

    if prediction > 3:
        score += 2
    elif prediction < 1:
        score -= 2

    if price < comp_price:
        score += 1
    else:
        score -= 1

    if discount > 0.3:
        score -= 1

    # Final decision
    if score >= 2:
        recommendations.append("🚀 Strong market → Increase price & scale")
    elif score == 1:
        recommendations.append("📈 Moderate growth → Slight price increase")
    elif score == 0:
        recommendations.append("⚖️ Stable → Maintain pricing")
    else:
        recommendations.append("⚠️ Risk → Reduce price or strategy change")

    # =========================
    # STEP 10: Price Optimization
    # =========================
    best_price = price

    if prediction < 1:
        best_price = price - 20
    elif prediction > 3:
        best_price = price + 20

    recommendations.append(f"Suggested optimal price: {best_price}")

    # =========================
    # STEP 11: Revenue Estimate
    # =========================
    revenue = best_price * prediction
    recommendations.append(f"Estimated revenue: {round(revenue, 2)}")

    # =========================
    # STEP 12: Final Response
    # =========================
    return jsonify({
        "predicted_demand": float(prediction),
        "demand_level": demand_level,
        "recommendations": recommendations
    })


# Run app
if __name__ == "__main__":
    app.run(port=5000, debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [20/May/2026 09:55:02] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [20/May/2026 09:55:02] "GET /favicon.ico HTTP/1.1" 404 -
